In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Cell 1 — define the cart payload schema

In [0]:
dj_cart_payload_schema = StructType([
    StructField("id", IntegerType()),
    StructField("userId", IntegerType()),
    StructField("total", DoubleType()),
    StructField("discountedTotal", DoubleType()),
    StructField("totalProducts", IntegerType()),
    StructField("totalQuantity", IntegerType()),
    StructField("products", ArrayType(StructType([
        StructField("id", IntegerType()),
        StructField("title", StringType()),
        StructField("price", DoubleType()),
        StructField("quantity", IntegerType()),
        StructField("total", DoubleType()),
        StructField("discountPercentage", DoubleType()),
        StructField("discountedTotal", DoubleType()),
    ]))),
])

### Cell 2 — parse, then explode the products array

In [0]:
dj_carts_parsed = (
    spark.table("bronze_dummyjson_carts"))
dj_carts_parsed.limit(1).select(col('raw_payload')).display()

raw_payload
"{""id"": 1, ""products"": [{""id"": 162, ""title"": ""Blue Frock"", ""price"": 29.99, ""quantity"": 4, ""total"": 119.96, ""discountPercentage"": 12.13, ""discountedTotal"": 105.41, ""thumbnail"": ""https://cdn.dummyjson.com/product-images/tops/blue-frock/thumbnail.webp""}, {""id"": 113, ""title"": ""Generic Motorcycle"", ""price"": 3999.99, ""quantity"": 3, ""total"": 11999.97, ""discountPercentage"": 12.1, ""discountedTotal"": 10547.97, ""thumbnail"": ""https://cdn.dummyjson.com/product-images/motorcycle/generic-motorcycle/thumbnail.webp""}, {""id"": 122, ""title"": ""iPhone 6"", ""price"": 299.99, ""quantity"": 3, ""total"": 899.97, ""discountPercentage"": 6.69, ""discountedTotal"": 839.76, ""thumbnail"": ""https://cdn.dummyjson.com/product-images/smartphones/iphone-6/thumbnail.webp""}, {""id"": 138, ""title"": ""Baseball Ball"", ""price"": 8.99, ""quantity"": 2, ""total"": 17.98, ""discountPercentage"": 1.71, ""discountedTotal"": 17.67, ""thumbnail"": ""https://cdn.dummyjson.com/product-images/sports-accessories/baseball-ball/thumbnail.webp""}], ""total"": 13037.88, ""discountedTotal"": 11510.81, ""userId"": 1, ""totalProducts"": 4, ""totalQuantity"": 12}"


In [0]:
dj_carts_parsed = (
    spark.table("bronze_dummyjson_carts")
    .withColumn("parsed", from_json(col("raw_payload"), dj_cart_payload_schema))
    .select(
        col("parsed.id").alias("cart_id"),
        col("parsed.userId").alias("customer_key"),
        explode(col("parsed.products")).alias("line_item"),
    )
)

dj_cart_lines = dj_carts_parsed.select(
    col("cart_id"),
    col("customer_key").cast("string"),
    col("line_item.id").alias("product_id_ref"),
    col("line_item.title").alias("product_title"),
    col("line_item.price").alias("unit_price"),
    col("line_item.quantity").alias("quantity"),
    col("line_item.total").alias("line_total"),
    col("line_item.discountPercentage").alias("discount_percentage"),
    col("line_item.discountedTotal").alias("line_discounted_total"),
)

display(dj_cart_lines.limit(10))
print(f"Cart line-item rows: {dj_cart_lines.count()}")

cart_id,customer_key,product_id_ref,product_title,unit_price,quantity,line_total,discount_percentage,line_discounted_total
1,1,162,Blue Frock,29.99,4,119.96,12.13,105.41
1,1,113,Generic Motorcycle,3999.99,3,11999.97,12.1,10547.97
1,1,122,iPhone 6,299.99,3,899.97,6.69,839.76
1,1,138,Baseball Ball,8.99,2,17.98,1.71,17.67
2,2,86,Man Short Sleeve Shirt,19.99,5,99.94999999999999,6.83,93.12
2,2,104,Apple iPhone Charger,19.99,2,39.98,18.52,32.58
3,3,24,Fish Steak,14.99,1,14.99,4.23,14.36
3,3,123,iPhone 13 Pro,1099.99,1,1099.99,9.37,996.92
3,3,129,Realme X,299.99,1,299.99,6.95,279.14
3,3,86,Man Short Sleeve Shirt,19.99,5,99.94999999999999,6.83,93.12


Cart line-item rows: 800


### Cell 3 — create silver_carts table with CDF enabled

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS mia_catalog.silver.silver_order_lines (
        cart_id INT,
        customer_key STRING,
        product_id_ref INT,
        product_title STRING,
        unit_price DOUBLE,
        quantity INT,
        line_total DOUBLE,
        discount_percentage DOUBLE,
        line_discounted_total DOUBLE,
        record_hash STRING,
        last_updated_ts TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print("mia_catalog.silver.silver_order_lines ready, CDF enabled")

mia_catalog.silver.silver_order_lines ready, CDF enabled


### Cell 4 — compute record_hash + timestamp

In [0]:
silver_order_lines_with_hash = (
    dj_cart_lines
    .withColumn(
        "record_hash",
        md5(concat_ws("|",
            col("cart_id"), col("product_id_ref"), col("quantity"),
            col("unit_price"), col("line_total"), col("discount_percentage")
        ))
    )
    .withColumn("last_updated_ts", current_timestamp())
)

### Cell 5 — register as temp view, then MERGE via SQL

In [0]:
silver_order_lines_with_hash.createOrReplaceTempView("silver_order_lines_updates")

spark.sql("""
    MERGE INTO mia_catalog.silver.silver_order_lines AS target
    USING silver_order_lines_updates AS source
    ON target.cart_id = source.cart_id AND target.product_id_ref = source.product_id_ref
    WHEN MATCHED AND target.record_hash != source.record_hash THEN
        UPDATE SET *
    WHEN NOT MATCHED THEN
        INSERT *
""")

print("MERGE completed")

MERGE completed


### Cell 6 — verify

In [0]:
display(spark.sql("SELECT count(*) FROM mia_catalog.silver.silver_order_lines"))

count(*)
800
